# PXDesign on Colab

> **Version 1.0.3.** The current version is always the one at
> [the releases page](https://github.com/a-LittleMonk/PXDesign/releases).
> If that page shows a higher number than the line you are reading, this copy
> is out of date — [open the current notebook](https://colab.research.google.com/github/a-LittleMonk/PXDesign/blob/main/PXDesign_colab.ipynb)
> and *File > Save a copy in Drive* to replace it. A saved copy never updates itself.

Protein design evaluation: takes designed backbones, puts sequences on them with
ProteinMPNN, folds those sequences back with AlphaFold2 / Protenix / ESMFold, and
scores how well the refold reproduces the design. Run steps 1-6 once (~25 min),
then use step 7 for evaluations.

**First: Runtime -> Change runtime type -> T4 GPU.**

<small>**Using Run all?** Two things need you: step 2 waits for a Google Drive
authorisation popup, and step 7's fields must already say what you want evaluated —
its defaults reproduce `binder_eval_demo.sh` on the five bundled PD-L1 binder
examples and ignore anything in Drive. Set those two up and the rest, including
saving back to Drive, runs unattended. The setup steps raise on a bad environment,
so a Run all stops at the real cause instead of cascading.</small>

<small>Runs [PXDesignBench](https://github.com/bytedance/PXDesignBench) unmodified,
inside its own Python 3.11 environment, called as a program rather than imported,
because Colab's own Python is too new for the pinned PyTorch / JAX this stack needs.
The evaluation and prediction logic is upstream's — this notebook only builds the
environment, fetches weights, and hands `pxdbench/run.py` the same arguments the
demo scripts do. Double-click a step title to see its code.</small>

In [ ]:
#@title 1. Check the GPU
#@markdown Must show a GPU. If it does not, fix the runtime type before going
#@markdown further: *Runtime > Change runtime type > T4 GPU*.
#@markdown
#@markdown Also reports whether the card can do bfloat16, which decides the
#@markdown Protenix dtype step 7 should use.

NOTEBOOK_VERSION = "1.0.3"
print(f"PXDesign on Colab - notebook version {NOTEBOOK_VERSION}\n")

!nvidia-smi || echo "nvidia-smi found no GPU"

try:
    import torch
    assert torch.cuda.is_available()
except Exception as exc:
    raise RuntimeError(
        "No GPU visible. Runtime > Change runtime type > T4 GPU, then run this "
        "cell again. Nothing below works on CPU."
    ) from exc

_name = torch.cuda.get_device_name(0)
_major, _minor = torch.cuda.get_device_capability(0)
_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
BF16_OK = _major >= 8

print(f"\nGPU         : {_name}")
print(f"compute     : {_major}.{_minor}")
print(f"memory      : {_mem:.1f} GB")
print(f"bfloat16    : {'yes' if BF16_OK else 'no - keep dtype at fp32 in step 7'}")
if _mem < 14:
    print("\nUnder 14 GB of VRAM. Protenix on a long target may run out of memory;")
    print("evaluate fewer sequences at a time, or turn the Protenix filter off.")

In [ ]:
#@title 2. Mount your Google Drive
#@markdown Gives the notebook a folder on your Drive, so you can feed it designs
#@markdown you keep there, have the results land back there, and keep the model
#@markdown weights between sessions instead of downloading them again.
#@markdown A browser popup will ask you to authorise access.
drive_folder = "PXDesign" #@param {type:"string"}

import os
from datetime import datetime
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT    = f"/content/drive/MyDrive/{drive_folder}"
DRIVE_INPUTS  = f"{DRIVE_ROOT}/inputs"
DRIVE_OUTPUT  = f"{DRIVE_ROOT}/outputs"
DRIVE_WEIGHTS = f"{DRIVE_ROOT}/tool_weights"
for _d in (DRIVE_INPUTS, DRIVE_OUTPUT, DRIVE_WEIGHTS):
    os.makedirs(_d, exist_ok=True)

with open("/content/.pxd_drive", "w") as f:
    f.write(DRIVE_ROOT)

print(f"Drive mounted at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Put designed structures in:  My Drive/{drive_folder}/inputs")
print(f"Results will be saved to:    My Drive/{drive_folder}/outputs")
print(f"Weights are cached in:       My Drive/{drive_folder}/tool_weights\n")

found = sorted(d for d in os.listdir(DRIVE_INPUTS)
               if os.path.isdir(os.path.join(DRIVE_INPUTS, d)))
loose = sorted(f for f in os.listdir(DRIVE_INPUTS)
               if f.lower().endswith((".pdb", ".cif")))
if found:
    print("Subfolders in your inputs folder - paste one of these names into")
    print("`drive_subfolder` in step 7:")
    for d in found:
        n = len([f for f in os.listdir(os.path.join(DRIVE_INPUTS, d))
                 if f.lower().endswith((".pdb", ".cif"))])
        print(f"    {d}   ({n} structures)")
if loose:
    print(f"\n{len(loose)} structure(s) sit directly in inputs/ - leave")
    print("`drive_subfolder` empty in step 7 to use those.")
if not found and not loose:
    print("inputs folder is empty for now. A whole evaluation runs on a *folder* of")
    print("designs, so drop your .pdb or .cif files in there (all against the same")
    print("target, same chain naming) and re-run this cell to list them.")

In [ ]:
#@title 3. Install Python 3.11 (~1 min)
#@markdown Sits alongside Colab's own Python without disturbing it. PXDesignBench
#@markdown pins PyTorch 2.3.1 and JAX 0.4.29, and those have no wheels for the
#@markdown Python Colab ships. Also installs `rsync`, which steps 6 and 10 use to
#@markdown move weights and results to and from Drive without recopying what is
#@markdown already there.

!apt-get update -qq
!apt-get install -y -qq python3.11 python3.11-venv python3.11-dev rsync
!python3.11 --version
!rsync --version | head -1

In [ ]:
#@title 4. Build the environment (~12 min, ~12 GB)
#@markdown Installs exactly what upstream's `install.sh` installs, in its order, into
#@markdown `/content/pxdenv`: PyTorch 2.3.1 (cu121), Protenix `v0.5.0+pxd`, ColabDesign,
#@markdown JAX 0.4.29 with CUDA, NumPy 1.26.3. Then clones PXDesignBench and installs it.
#@markdown Let it finish - do not interrupt it or re-run it part way through.
#@markdown
#@markdown Ends by checking itself and **raising** if anything is wrong, so a Run all
#@markdown stops here rather than failing later for a misleading reason.
pxdbench_ref = "main" #@param {type:"string"}
#@markdown <small>`pxdbench_ref` is the PXDesignBench branch, tag or commit to install.
#@markdown Leave it at `main` to track upstream; pin a commit if you need a run to be
#@markdown reproducible months from now.</small>

import os, subprocess, textwrap

ENV  = "/content/pxdenv"
PY   = f"{ENV}/bin/python"
PIP  = f"{ENV}/bin/pip"
REPO = "/content/PXDesignBench"

# A half-built environment is worse than none: rebuild rather than install on top.
if not os.path.isfile(PY):
    print("creating environment...")
    os.system(f"rm -rf {ENV}")
    os.system(f"python3.11 -m venv {ENV}")
assert os.path.isfile(PY), "venv creation failed - check step 3's output"

# Colab exports PYTHONPATH=/env/python, which would leak Colab's own packages
# (NumPy 2, a different torch) into this environment. Every subprocess below is
# launched with it replaced by the repo itself.
PXD_ENV = {k: v for k, v in os.environ.items() if k != "PYTHONPATH"}
PXD_ENV["PATH"] = f"{ENV}/bin:" + os.environ.get("PATH", "")
PXD_ENV["PYTHONPATH"] = REPO
PXD_ENV["PYTHONNOUSERSITE"] = "1"
PXD_ENV["VIRTUAL_ENV"] = ENV

!{PIP} install -q --upgrade pip setuptools wheel
!{PIP} install -q ninja

# 1) GPU PyTorch first: deepspeed, pulled in by Protenix below, imports torch while
#    it installs itself and fails outright if torch is not already there.
!{PIP} install -q "torch==2.3.1" --index-url https://download.pytorch.org/whl/cu121

# 2) Protenix - the structure predictor PXDesignBench filters with.
!DS_BUILD_OPS=0 {PIP} install -q "git+https://github.com/bytedance/Protenix.git@v0.5.0+pxd"

# 3) PXDesignBench's own dependencies.
!{PIP} install -q einops natsort dm-tree posix_ipc "transformers==4.51.3" "dm-haiku==0.0.13" "optax==0.2.5"

# 4) ColabDesign, which is how AlphaFold2 is run here. --no-deps on purpose: its
#    dependency list would drag in a different JAX than the one pinned below.
!{PIP} install -q git+https://github.com/sokrypton/ColabDesign.git --no-deps

# 5) JAX with CUDA. The wheel on jax_cuda_releases bundles its own CUDA and cuDNN,
#    so it does not fight PyTorch over the nvidia-* pip packages. If that index is
#    ever pulled, fall back to the plugin build from PyPI.
!{PIP} install -q "jax[cuda]==0.4.29" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
_gpu_jax = subprocess.run(
    [PY, "-c", "import jax, sys; "
               "sys.exit(0 if any(d.platform == 'gpu' for d in jax.devices()) else 1)"],
    env=PXD_ENV, capture_output=True)
if _gpu_jax.returncode != 0:
    print("\nCUDA jaxlib not installed from jax_cuda_releases - falling back to jax[cuda12]")
    !{PIP} install -q "jax[cuda12]==0.4.29"

# 6) Compatibility pins. pip resolves these to their newest releases, which now
#    expect jax 0.7 and NumPy 2 and would fail at import under the pins above:
#    chex arrives through optax, and ColabDesign reaches it through optax too.
#    immutabledict is declared by ColabDesign but never imported - installed anyway
#    so the dependency warning does not mask a real one.
!{PIP} install -q "chex==0.1.87" "ml_dtypes==0.4.0" immutabledict

# 7) Pinned last so nothing drags NumPy 2 back in: this stack was built against 1.x.
!{PIP} install -q "numpy==1.26.3"

# 8) PXDesignBench itself.
if not os.path.isdir(REPO):
    !git clone -q https://github.com/bytedance/PXDesignBench.git {REPO}
!cd {REPO} && git fetch -q origin "+refs/heads/*:refs/remotes/origin/*" --tags 2>/dev/null; cd {REPO} && git checkout -q {pxdbench_ref} 2>/dev/null || git checkout -q -B {pxdbench_ref} origin/{pxdbench_ref} 2>/dev/null; cd {REPO} && git pull -q --ff-only 2>/dev/null; true
!{PIP} install -q -e {REPO}
!chmod +x {REPO}/pxdbench/metrics/TMalign

_head = subprocess.run(["git", "-C", REPO, "rev-parse", "--short", "HEAD"],
                       capture_output=True, text=True).stdout.strip()

print("\n--- environment check ---")
_check = textwrap.dedent('''
    import sys, torch, numpy, jax, jaxlib, colabdesign, protenix, pxdbench, transformers
    import chex, optax
    # What step 7 actually imports: AlphaFold2 and ProteinMPNN both come through
    # these, and they reach jax via optax/chex rather than directly.
    from colabdesign import clear_mem, mk_afdesign_model
    from colabdesign.mpnn import mk_mpnn_model
    print('python  :', sys.version.split()[0])
    print('torch   :', torch.__version__)
    print('numpy   :', numpy.__version__)
    print('jax     :', jax.__version__)
    print('jaxlib  :', jaxlib.__version__)
    print('cuda    :', torch.cuda.is_available())
    print('gpu     :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
    print('jaxdev  :', jax.devices()[0].platform)
    print('torch<->numpy:', torch.tensor([1.0]).numpy().sum())
    from pxdbench.tasks import get_task_class
    print('chex    :', chex.__version__)
    print('af2+mpnn: importable')
    print('tasks   :', get_task_class('binder').__name__, get_task_class('monomer').__name__)
''')
with open("/content/_check.py", "w") as f:
    f.write(_check)
_chk = subprocess.run([PY, "-u", "/content/_check.py"], env=PXD_ENV,
                      capture_output=True, text=True)
print(_chk.stdout + _chk.stderr)

# Verified rather than left for the eye, so an unattended Run all halts on the real
# cause instead of carrying a broken environment into the steps below.
_vals = {k.strip(): v.strip()
         for k, v in (l.split(":", 1) for l in _chk.stdout.splitlines() if ":" in l)}
_bad = []
if _chk.returncode:
    _bad.append("the check script itself failed - see its output above")
if not _vals.get("numpy", "").startswith("1."):
    _bad.append(f"numpy is {_vals.get('numpy', '?')}, must be 1.26.3 - something "
                "reinstalled NumPy 2 over the pin")
if not _vals.get("torch", "").startswith("2.3.1"):
    _bad.append(f"torch is {_vals.get('torch', '?')}, must be 2.3.1")
if _vals.get("cuda") != "True":
    _bad.append("torch cannot see the GPU - check the runtime type (step 1)")
if _vals.get("af2+mpnn") != "importable":
    _bad.append("ColabDesign will not import, so AlphaFold2 and ProteinMPNN cannot "
                "run - usually chex or ml_dtypes resolved to a release that wants "
                "jax 0.7 / NumPy 2")
if _vals.get("jaxdev") != "gpu":
    _bad.append("JAX is on CPU, so AlphaFold2 would take hours per design - the "
                "CUDA jaxlib did not install")

if _bad:
    print("\n*** ENVIRONMENT IS NOT USABLE ***")
    for _b in _bad:
        print("  - " + _b)
    raise RuntimeError(
        f"Environment check failed ({len(_bad)} problem(s), listed above). "
        "Do NOT re-run this cell on top of the failure: Runtime > Disconnect and "
        "delete runtime, then start again from step 1.")

print(f"\nEnvironment OK. PXDesignBench at {_head} ({pxdbench_ref}).")

**Check step 4's output before continuing.** You want `torch: 2.3.1+cu121`,
`numpy: 1.26.3`, `jaxdev: gpu`, `cuda: True`, a GPU name, and a `torch<->numpy` line
printing `1.0`. If JAX came out on CPU, stop here — AlphaFold2 would take hours per
design rather than a minute, and the run would look like a hang rather than an error.

In [ ]:
#@title 5. Compile the fused LayerNorm kernel (~3 min)
#@markdown Protenix ships a fused LayerNorm CUDA kernel and uses it when
#@markdown `LAYERNORM_TYPE=fast_layernorm` is set, which is what upstream's demo
#@markdown scripts set. It is a speed optimisation: the same arithmetic as
#@markdown `torch.nn.LayerNorm`, fused into one kernel. Compiling it here means the
#@markdown first evaluation does not stall for minutes doing it.
#@markdown
#@markdown If it fails to build, this cell says so and step 7 simply leaves the
#@markdown variable unset, which falls back to PyTorch's own LayerNorm. Results are
#@markdown the same to within float noise; runs are somewhat slower.

import os, subprocess

ENV, REPO = "/content/pxdenv", "/content/PXDesignBench"
PY = f"{ENV}/bin/python"
assert os.path.isfile(PY), "run step 4 first"

PXD_ENV = {k: v for k, v in os.environ.items() if k != "PYTHONPATH"}
PXD_ENV.update({"PATH": f"{ENV}/bin:" + os.environ.get("PATH", ""),
                "PYTHONPATH": REPO, "PYTHONNOUSERSITE": "1", "VIRTUAL_ENV": ENV})

print("compiling fastfold_layer_norm_cuda (first time only)...")
_ln = subprocess.run(
    [PY, "-c", "from protenix.model.layer_norm.layer_norm import FusedLayerNorm; "
               "print('fast_layernorm OK')"],
    env=PXD_ENV, capture_output=True, text=True)
print(_ln.stdout[-2000:])

FASTLN_OK = "fast_layernorm OK" in _ln.stdout
flag = "/content/.pxd_fastln_ok"
if FASTLN_OK:
    open(flag, "w").close()
    print("Fused LayerNorm available - step 7 will use it, as the demo scripts do.")
else:
    if os.path.exists(flag):
        os.remove(flag)
    print(_ln.stderr[-3000:])
    print("\nThe kernel did not build. This is not fatal: step 7 will run with")
    print("PyTorch's own LayerNorm instead, which is numerically equivalent and a")
    print("little slower. Carry on to step 6.")

In [ ]:
#@title 6. Download model weights
#@markdown The weights are not bundled with the code. Which ones you need depends on
#@markdown the task: the **binder** pipeline folds with AlphaFold2 and Protenix, the
#@markdown **monomer** pipeline folds with ESMFold. ProteinMPNN is fetched either way —
#@markdown it is small, and the monomer pipeline reads it from here (the binder
#@markdown pipeline uses the copy inside ColabDesign). Protenix fetches its own
#@markdown checkpoint and CCD files on the first evaluation.
#@markdown
#@markdown By default nothing touches your Drive: the weights are downloaded to
#@markdown Colab's own disk and go away with the runtime. Tick `cache_in_drive` to
#@markdown keep them — it then looks in Drive first and uses what is already there,
#@markdown and only downloads and saves what is missing.
weights_for = "binder (AlphaFold2 + ProteinMPNN)" #@param ["binder (AlphaFold2 + ProteinMPNN)", "monomer (ESMFold + ProteinMPNN)", "both"]
af2_params = "model 1 only (~1.1 GB)" #@param ["model 1 only (~1.1 GB)", "all five models (~5.3 GB)"]
cache_in_drive = False #@param {type:"boolean"}
#@markdown <small>`model 1 only` is all the default settings use — upstream's
#@markdown `af2.model_ids` is `[0]`. Take all five only if you raise `af2_model_ids`
#@markdown in step 7. Either way the full 5.3 GB archive is downloaded and read; the
#@markdown choice is how much of it is kept on disk.</small>

import os, shutil, subprocess

ENV, REPO = "/content/pxdenv", "/content/PXDesignBench"
PY = f"{ENV}/bin/python"
WEIGHTS = "/content/tool_weights"
AF2_DIR, MPNN_DIR, ESM_DIR = (f"{WEIGHTS}/af2", f"{WEIGHTS}/mpnn", f"{WEIGHTS}/esmfold")
assert os.path.isfile(PY), "run step 4 first"
for _d in (AF2_DIR, MPNN_DIR, ESM_DIR):
    os.makedirs(_d, exist_ok=True)

need_af2 = weights_for.startswith(("binder", "both"))
need_esm = weights_for.startswith(("monomer", "both"))

DRIVE_WEIGHTS = None
if cache_in_drive and os.path.isfile("/content/.pxd_drive"):
    DRIVE_WEIGHTS = open("/content/.pxd_drive").read().strip() + "/tool_weights"
    os.makedirs(DRIVE_WEIGHTS, exist_ok=True)
    print(f"Drive cache on: {DRIVE_WEIGHTS.replace('/content/drive/MyDrive', 'My Drive')}")
    print("Looking there first; only what is missing is downloaded and saved back.\n")
elif cache_in_drive:
    print("Drive is not mounted (step 2) - downloading to local disk only.\n")
else:
    print("Drive cache off: downloading to Colab's disk only, nothing is written to")
    print("your Drive. These weights go away when the runtime ends - tick")
    print("`cache_in_drive` if you would rather keep them.\n")


def restore(sub):
    # Copy a weights subfolder back from Drive, if a previous session left one.
    if not DRIVE_WEIGHTS:
        return
    src = os.path.join(DRIVE_WEIGHTS, sub)
    if os.path.isdir(src) and os.listdir(src):
        print(f"restoring {sub} from Drive...")
        subprocess.run(["rsync", "-a", src + "/", os.path.join(WEIGHTS, sub) + "/"])


def stash(sub):
    if not DRIVE_WEIGHTS:
        return
    print(f"copying {sub} to Drive...")
    subprocess.run(["rsync", "-a", os.path.join(WEIGHTS, sub) + "/",
                    os.path.join(DRIVE_WEIGHTS, sub) + "/"])


# ---- AlphaFold2 parameters -------------------------------------------------
if need_af2:
    restore("af2")
    want = ["params_model_1.npz", "params_model_1_ptm.npz"]
    if af2_params.startswith("all five"):
        want = [f"params_model_{i}{s}.npz" for i in range(1, 6)
                for s in ("", "_ptm", "_multimer_v3")]
    missing = [w for w in want if not os.path.isfile(f"{AF2_DIR}/{w}")]
    if missing:
        url = "https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar"
        # Piped straight into tar, selecting members: the archive is 5.3 GB and
        # there is no reason to land it on disk before throwing most of it away.
        members = ("--wildcards 'params_model_1*.npz' LICENSE"
                   if not af2_params.startswith("all five") else "")
        print(f"downloading AlphaFold2 parameters ({len(missing)} file(s) missing)...")
        !curl -sL {url} | tar -x -C {AF2_DIR} {members} || true
        stash("af2")
    else:
        print("AlphaFold2 parameters already present.")

# ---- ProteinMPNN weights ---------------------------------------------------
restore("mpnn")
if not os.path.isdir(f"{MPNN_DIR}/vanilla_model_weights"):
    print("downloading ProteinMPNN weights...")
    !rm -rf /content/_mpnn && git clone -q --depth 1 https://github.com/dauparas/ProteinMPNN.git /content/_mpnn
    for sub in ("ca_model_weights", "soluble_model_weights", "vanilla_model_weights"):
        if not os.path.isdir(f"{MPNN_DIR}/{sub}"):
            shutil.copytree(f"/content/_mpnn/{sub}", f"{MPNN_DIR}/{sub}")
    shutil.rmtree("/content/_mpnn", ignore_errors=True)
    stash("mpnn")
else:
    print("ProteinMPNN weights already present.")

# ---- ESMFold ---------------------------------------------------------------
if need_esm:
    restore("esmfold")
    if not os.path.isfile(f"{ESM_DIR}/pytorch_model.bin"):
        print("downloading ESMFold (~2.7 GB)...")
        !{PY} -c "from huggingface_hub import snapshot_download; snapshot_download('facebook/esmfold_v1', local_dir='{ESM_DIR}', allow_patterns=['*.json','*.txt','*.bin','*.model'])"
        stash("esmfold")
    else:
        print("ESMFold weights already present.")

# ---- verify ---------------------------------------------------------------
print("\n--- weights check ---")
required = []
if need_af2:
    required += [f"{AF2_DIR}/params_model_1.npz", f"{AF2_DIR}/params_model_1_ptm.npz"]
if need_esm:
    required += [f"{ESM_DIR}/pytorch_model.bin", f"{ESM_DIR}/config.json"]
required += [f"{MPNN_DIR}/vanilla_model_weights/v_48_020.pt",
             f"{MPNN_DIR}/ca_model_weights/v_48_020.pt"]

missing = [p for p in required if not os.path.isfile(p)]
for p in required:
    if os.path.isfile(p):
        print(f"ok    {p}  ({os.path.getsize(p) / 1e6:.0f} MB)")
    else:
        print(f"MISS  {p}")
if missing:
    raise RuntimeError(
        f"{len(missing)} weight file(s) did not download. Most often this is a full "
        "disk - check the resources pane - or a dropped connection. Re-run this cell; "
        "what is already there is kept.")
print(f"\nWeights OK, in {WEIGHTS}. TOOL_WEIGHTS_ROOT will point here.")

In [ ]:
#@title 7. Run the evaluation
#@markdown Hands `pxdbench/run.py` (binder) or `pxdbench/run_monomer.py` (monomer)
#@markdown the arguments below and streams its output. Nothing here reimplements any
#@markdown of the pipeline — every setting is an upstream config key, named as upstream
#@markdown names it.

#@markdown ---
#@markdown ### What to evaluate
task = "binder" #@param ["binder", "monomer"]
demo_settings = True #@param {type:"boolean"}
#@markdown <small>`demo_settings` reproduces upstream's `binder_eval_demo.sh` /
#@markdown `monomer_eval_demo.sh` exactly — bundled examples, their chains, their
#@markdown sequence counts — and ignores every field below except `run_name` and
#@markdown `protenix_dtype`. Untick it to evaluate your own designs.</small>
run_name = "run1" #@param {type:"string"}
inputs_from = "bundled example" #@param ["bundled example", "Google Drive folder", "path on disk"]
drive_subfolder = "" #@param {type:"string"}
disk_path = "" #@param {type:"string"}
is_mmcif = True #@param {type:"boolean"}
file_names = "" #@param {type:"string"}
seed = 2025 #@param {type:"integer"}
#@markdown <small>`is_mmcif` says whether the folder holds `.cif` (ticked) or `.pdb`
#@markdown (unticked) — all of them, not a mix. `file_names` restricts the run to a
#@markdown comma-separated list of names without extension; empty means the whole
#@markdown folder.</small>

#@markdown ---
#@markdown ### Chains (binder task)
binder_chains = "B0" #@param {type:"string"}
cond_chains = "" #@param {type:"string"}
orig_seqs_json = "examples/orig_seqs_test.json" #@param {type:"string"}
#@markdown <small>One binder chain only — upstream asserts that. Every other chain is
#@markdown the target unless you name `cond_chains` yourself. `orig_seqs_json` is
#@markdown optional metadata for the target: the uncropped sequence, and a
#@markdown precomputed MSA so Protenix does not have to call the MSA server. Leave it
#@markdown empty and the server is called instead, which needs a few minutes and a
#@markdown working connection.</small>

#@markdown ---
#@markdown ### Sequence design (ProteinMPNN)
num_seqs = 2 #@param {type:"integer"}
mpnn_temperature = "0.0001" #@param {type:"string"}
mpnn_binder_weights = "original" #@param ["original", "soluble"]
mpnn_monomer_model = "ca" #@param ["ca", "bb", "soluble"]
mpnn_rm_aa = "C" #@param {type:"string"}
mpnn_fix_interface = False #@param {type:"boolean"}
mpnn_bias_aa_json = "" #@param {type:"string"}
use_gt_seq = False #@param {type:"boolean"}
#@markdown <small>`num_seqs` sequences are designed per input structure. `use_gt_seq`
#@markdown skips ProteinMPNN and refolds the sequence already in the structure file.
#@markdown `mpnn_bias_aa_json` biases the amino acid composition — see
#@markdown `examples/hydrophilic_bias.json` in the repo.</small>

#@markdown ---
#@markdown ### Which filters to run (binder task)
eval_complex_af2 = True #@param {type:"boolean"}
eval_binder_monomer_af2 = True #@param {type:"boolean"}
eval_protenix_mini = True #@param {type:"boolean"}
eval_protenix_base = False #@param {type:"boolean"}
eval_diversity = False #@param {type:"boolean"}
#@markdown <small>Each one costs its own fold per sequence. `eval_protenix_base` is the
#@markdown big Protenix model and is off upstream too. `eval_diversity` is pairwise
#@markdown TM-score across the designs, on CPU.</small>

#@markdown ---
#@markdown ### AlphaFold2
af2_use_multimer = False #@param {type:"boolean"}
af2_model_ids = "0" #@param {type:"string"}
af2_use_initial_guess = True #@param {type:"boolean"}
af2_use_binder_template = True #@param {type:"boolean"}
af2_use_initial_atom_pos = False #@param {type:"boolean"}
af2_is_cyclic = False #@param {type:"boolean"}

#@markdown ---
#@markdown ### Protenix
protenix_dtype = "fp32" #@param ["fp32", "bf16"]
ptx_n_cycle = 4 #@param {type:"integer"}
ptx_n_sample = 1 #@param {type:"integer"}
ptx_n_step = 2 #@param {type:"integer"}
ptx_use_msa = True #@param {type:"boolean"}
#@markdown <small>Keep `fp32` on a T4 — it has no bfloat16. DeepSpeed's evoformer
#@markdown attention stays off either way; it needs a build this environment does not
#@markdown have, and upstream's demos disable it too.</small>

import os, re, shutil, subprocess, sys, time

ENV, REPO = "/content/pxdenv", "/content/PXDesignBench"
PY, WEIGHTS = f"{ENV}/bin/python", "/content/tool_weights"
assert os.path.isfile(PY), "run step 4 first"
assert os.path.isdir(f"{WEIGHTS}/mpnn/vanilla_model_weights"), "run step 6 first"

# Upstream's config parser zips argv into `--key value` pairs and re-splits the
# result on whitespace, so no value may contain a space and none may be empty.
# Everything therefore runs out of space-free paths under /content, and Drive is
# only ever read from and written to by this notebook, never by the subprocess.
WORK = "/content/pxd_work"
OUT_ROOT = "/content/pxd_output"
safe_run = re.sub(r"[^A-Za-z0-9._-]", "_", run_name) or "run1"
out_dir = f"{OUT_ROOT}/{safe_run}"

if demo_settings:
    # Exactly binder_eval_demo.sh / monomer_eval_demo.sh.
    if task == "binder":
        inputs_from, is_mmcif, file_names = "bundled example", True, ""
        binder_chains, cond_chains = "B0", ""
        orig_seqs_json = "examples/orig_seqs_test.json"
        num_seqs, mpnn_temperature, use_gt_seq = 2, "0.0001", False
        mpnn_binder_weights, mpnn_rm_aa, mpnn_fix_interface = "original", "C", False
        mpnn_bias_aa_json = ""
        eval_complex_af2 = eval_binder_monomer_af2 = eval_protenix_mini = True
        eval_protenix_base = eval_diversity = False
        af2_use_multimer, af2_model_ids = False, "0"
        af2_use_initial_guess = af2_use_binder_template = True
        af2_use_initial_atom_pos = af2_is_cyclic = False
        ptx_n_cycle, ptx_n_sample, ptx_n_step, ptx_use_msa = 4, 1, 2, True
    else:
        inputs_from, is_mmcif, file_names = "bundled example", False, ""
        num_seqs, mpnn_temperature, mpnn_monomer_model = 8, "0.1", "ca"
        eval_diversity, use_gt_seq, mpnn_bias_aa_json = False, False, ""
    seed = 2025
    print("Using upstream demo settings.\n")

# ---- where the designs come from -----------------------------------------
if inputs_from == "bundled example":
    data_dir = "examples/binder" if task == "binder" else "examples/monomer"
    abs_data = os.path.join(REPO, data_dir)
elif inputs_from == "Google Drive folder":
    assert os.path.isfile("/content/.pxd_drive"), "Drive is not mounted - run step 2"
    src = open("/content/.pxd_drive").read().strip() + "/inputs"
    if drive_subfolder.strip():
        src = os.path.join(src, drive_subfolder.strip())
    assert os.path.isdir(src), f"no such folder in Drive: {src}"
    # Copied onto local disk first: Drive paths contain a space ("My Drive"), which
    # upstream's argument parser cannot carry, and Drive reads are slow besides.
    abs_data = f"{WORK}/{safe_run}_inputs"
    shutil.rmtree(abs_data, ignore_errors=True)
    os.makedirs(abs_data)
    ext = ".cif" if is_mmcif else ".pdb"
    picked = [f for f in sorted(os.listdir(src)) if f.lower().endswith(ext)]
    assert picked, f"no {ext} files in {src} (is `is_mmcif` set the right way?)"
    for f in picked:
        shutil.copy2(os.path.join(src, f), abs_data)
    print(f"copied {len(picked)} {ext} file(s) from Drive\n")
    data_dir = abs_data
else:
    abs_data = data_dir = disk_path.strip()
    assert abs_data and os.path.isdir(abs_data), f"not a folder: {abs_data!r}"
    assert " " not in abs_data, "path must not contain spaces"

os.makedirs(out_dir, exist_ok=True)
b = lambda v: "true" if v else "false"

# ---- the command, keyed exactly as upstream's configs are ----------------
if task == "binder":
    cmd = [PY, "-u", "pxdbench/run.py",
           "--data_dir", data_dir,
           "--dump_dir", out_dir,
           "--is_mmcif", b(is_mmcif),
           "--seed", str(seed),
           "--binder_chains", binder_chains.strip(),
           "--binder.num_seqs", str(num_seqs),
           "--binder.use_gt_seq", b(use_gt_seq),
           "--binder.eval_diversity", b(eval_diversity),
           "--binder.eval_complex", b(eval_complex_af2),
           "--binder.eval_binder_monomer", b(eval_binder_monomer_af2),
           "--binder.eval_protenix_mini", b(eval_protenix_mini),
           "--binder.eval_protenix", b(eval_protenix_base),
           "--binder.tools.mpnn.temperature", mpnn_temperature.strip(),
           "--binder.tools.mpnn.weights", mpnn_binder_weights,
           "--binder.tools.mpnn.rm_aa", mpnn_rm_aa.strip() or "C",
           "--binder.tools.mpnn.fix_interface", b(mpnn_fix_interface),
           "--binder.tools.af2.use_multimer", b(af2_use_multimer),
           "--binder.tools.af2.model_ids", af2_model_ids.replace(" ", ""),
           "--binder.tools.af2.use_initial_guess", b(af2_use_initial_guess),
           "--binder.tools.af2.use_binder_template", b(af2_use_binder_template),
           "--binder.tools.af2.use_initial_atom_pos", b(af2_use_initial_atom_pos),
           "--binder.tools.af2.is_cyclic", b(af2_is_cyclic),
           "--binder.tools.ptx_mini.dtype", protenix_dtype,
           "--binder.tools.ptx_mini.use_deepspeed_evo_attention", "false",
           "--binder.tools.ptx_mini.N_cycle", str(ptx_n_cycle),
           "--binder.tools.ptx_mini.N_sample", str(ptx_n_sample),
           "--binder.tools.ptx_mini.N_step", str(ptx_n_step),
           "--binder.tools.ptx_mini.use_msa", b(ptx_use_msa),
           "--binder.tools.ptx.dtype", protenix_dtype,
           "--binder.tools.ptx.use_deepspeed_evo_attention", "false"]
    if cond_chains.strip():
        cmd += ["--cond_chains", cond_chains.replace(" ", "")]
    if orig_seqs_json.strip():
        cmd += ["--orig_seqs_json", orig_seqs_json.strip()]
    if mpnn_bias_aa_json.strip():
        cmd += ["--binder.tools.mpnn.bias_aa_json", mpnn_bias_aa_json.strip()]
    if file_names.strip():
        cmd += ["--file_name_list", file_names.replace(" ", "")]
else:
    cmd = [PY, "-u", "pxdbench/run_monomer.py",
           "--data_dir", data_dir,
           "--dump_dir", out_dir,
           "--is_mmcif", b(is_mmcif),
           "--seed", str(seed),
           "--monomer.num_seqs", str(num_seqs),
           "--monomer.tools.mpnn.temperature", mpnn_temperature.strip(),
           "--monomer.tools.mpnn.model_type", mpnn_monomer_model,
           "--monomer.eval_diversity", b(eval_diversity)]
    if mpnn_bias_aa_json.strip():
        cmd += ["--monomer.tools.mpnn.bias_aa_json", mpnn_bias_aa_json.strip()]

# ---- environment ---------------------------------------------------------
env = {k: v for k, v in os.environ.items() if k != "PYTHONPATH"}
env.update({
    # PATH first: ProteinMPNN, AlphaFold2 and Protenix each run in a subprocess that
    # upstream launches as plain `python3`, so `python3` has to mean this environment.
    "PATH": f"{ENV}/bin:" + os.environ.get("PATH", ""),
    "PYTHONPATH": REPO,
    "PYTHONNOUSERSITE": "1",
    "VIRTUAL_ENV": ENV,
    "TOOL_WEIGHTS_ROOT": WEIGHTS,
    "USE_DEEPSPEED_EVO_ATTENTION": "false",
    # JAX would otherwise reserve three quarters of the card up front and leave
    # nothing for the PyTorch half of the pipeline.
    "XLA_PYTHON_CLIENT_PREALLOCATE": "false",
    "XLA_PYTHON_CLIENT_MEM_FRACTION": "0.5",
})
if os.path.exists("/content/.pxd_fastln_ok"):
    env["LAYERNORM_TYPE"] = "fast_layernorm"

print("command:")
print("  " + " \\\n    ".join(cmd[1:]))
print(f"\ninput : {abs_data}")
print(f"output: {out_dir}")
print(f"fused LayerNorm: {'on' if 'LAYERNORM_TYPE' in env else 'off (torch LayerNorm)'}")
n_struct = len([f for f in os.listdir(abs_data)
                if f.lower().endswith(".cif" if is_mmcif else ".pdb")])
print(f"\n{n_struct} structure(s) x {num_seqs} sequence(s) to fold. The first")
print("evaluation also downloads the Protenix checkpoint and CCD files (~1 GB).")
print("-" * 70)

t0 = time.time()
proc = subprocess.Popen(cmd, cwd=REPO, env=env, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1,
                        encoding="utf-8", errors="replace")

# Protenix draws its download progress with carriage returns, which Colab turns
# into one line per redraw - thousands of them for a 1 GB checkpoint. Kept, but
# thinned to every 10%, so the real output stays readable.
_bar = re.compile(r"^\[[=\-]+\]\s+(\d+(?:\.\d+)?)%\s*$")
_last_pct = -100.0
for line in proc.stdout:
    m = _bar.match(line.strip())
    if m:
        pct = float(m.group(1))
        if pct - _last_pct < 10 and pct < 100:
            continue
        _last_pct = pct
    sys.stdout.write(line)
rc = proc.wait()
mins = (time.time() - t0) / 60

print("-" * 70)
if rc != 0:
    raise RuntimeError(
        f"Evaluation exited with code {rc} after {mins:.1f} min. The real error is "
        "in the output above - the last traceback, not this line. Out-of-memory on a "
        "T4 is the common one: turn off a filter, or lower num_seqs.")
print(f"Done in {mins:.1f} min. Results in {out_dir}")
print("Run step 10 before closing the tab - /content is deleted when the runtime ends.")

In [ ]:
#@title 8. Results
#@markdown Reads the two files upstream writes — `summary_output.json`, the averages
#@markdown and pass counts over the whole run, and `sample_level_output.csv`, one row
#@markdown per designed sequence.
run_name = "run1" #@param {type:"string"}
sort_by = "" #@param {type:"string"}
#@markdown <small>`sort_by` is any column of the CSV, best first — try `i_pTM` or
#@markdown `scRMSD`. Empty keeps upstream's order.</small>

import json, os, re, glob
import pandas as pd
from IPython.display import display

safe_run = re.sub(r"[^A-Za-z0-9._-]", "_", run_name) or "run1"
out_dir = f"/content/pxd_output/{safe_run}"
assert os.path.isdir(out_dir), f"no such run: {out_dir} - check run_name"

summaries = sorted(glob.glob(f"{out_dir}/**/summary_output.json", recursive=True))
samples = sorted(glob.glob(f"{out_dir}/**/sample_level_output.csv", recursive=True))
assert summaries or samples, f"{out_dir} holds no results - did step 7 finish?"

pd.set_option("display.max_columns", 200, "display.width", 250)

for s in summaries:
    with open(s) as f:
        summary = json.load(f)
    print("=" * 70)
    print(os.path.relpath(s, out_dir))
    print("=" * 70)
    # Pass counts first: they are the numbers the filters exist to produce.
    passes = {k: v for k, v in summary.items() if k.endswith("_success.count")}
    if passes:
        print("\ndesigns passing each filter")
        for k, v in sorted(passes.items()):
            print(f"  {k.replace('_success.count', '').rjust(12)} : {v}")
    print("\nsummary")
    for k, v in summary.items():
        if k in passes:
            continue
        print(f"  {k.rjust(40)} : {v if not isinstance(v, float) else round(v, 4)}")

for c in samples:
    df = pd.read_csv(c)
    print("\n" + "=" * 70)
    print(f"{os.path.relpath(c, out_dir)}  -  {len(df)} rows")
    print("=" * 70)
    if sort_by.strip():
        col = sort_by.strip()
        if col in df.columns:
            ascending = any(t in col.lower() for t in ("rmsd", "pae", "loss"))
            df = df.sort_values(col, ascending=ascending)
        else:
            print(f"(no column {col!r}; columns are: {', '.join(df.columns)})\n")
    display(df)

In [ ]:
#@title 9. View a prediction
#@markdown Overlays a refolded structure on the design it came from. Grey is the
#@markdown design that went in; coloured is what the folding model predicted from the
#@markdown designed sequence alone. The closer they sit, the better that design scored.
run_name = "run1" #@param {type:"string"}
prediction = 0 #@param {type:"integer"}
#@markdown <small>`prediction` indexes the list this cell prints. Run it once to see
#@markdown the list, then set the number.</small>

import glob, os, re

try:
    import py3Dmol
except ImportError:
    !pip install -q py3Dmol
    import py3Dmol

safe_run = re.sub(r"[^A-Za-z0-9._-]", "_", run_name) or "run1"
out_dir = f"/content/pxd_output/{safe_run}"
assert os.path.isdir(out_dir), f"no such run: {out_dir} - check run_name"

preds = sorted(p for p in glob.glob(f"{out_dir}/**/*.pdb", recursive=True)
               if "converted_pdbs" not in p)
assert preds, f"no predicted structures under {out_dir}"

print(f"{len(preds)} prediction(s):")
for i, p in enumerate(preds):
    print(f"  [{i}] {os.path.relpath(p, out_dir)}")

idx = max(0, min(prediction, len(preds) - 1))
pred_path = preds[idx]
print(f"\nshowing [{idx}] {os.path.relpath(pred_path, out_dir)}")

# The design the prediction is named after: strip the _seqN / _modelN suffixes
# upstream appends, then look for that name among the inputs.
stem = os.path.basename(pred_path)[:-4]
stem = re.sub(r"_model\d+$", "", stem)
stem = re.sub(r"_seed_\d+.*$", "", stem)
base = re.sub(r"_seq\d+$", "", stem)
design = None
for cand in glob.glob(f"/content/pxd_work/**/{base}.pdb", recursive=True) + \
            glob.glob(f"/content/PXDesignBench/examples/**/{base}.pdb", recursive=True) + \
            glob.glob(f"{out_dir}/**/converted_pdbs/{base}.pdb", recursive=True):
    design = cand
    break

view = py3Dmol.view(width=900, height=600)
view.addModel(open(pred_path).read(), "pdb")
view.setStyle({"model": 0}, {"cartoon": {"colorscheme": "chain"}})
if design:
    view.addModel(open(design).read(), "pdb")
    view.setStyle({"model": 1}, {"cartoon": {"color": "lightgrey", "opacity": 0.65}})
    print(f"design overlay: {design}")
else:
    print(f"(no input structure found for {base!r} - showing the prediction alone)")
view.zoomTo()
view.show()

In [ ]:
#@title 10. Save results to Drive
#@markdown Colab wipes `/content` when the runtime ends. Run this before closing the
#@markdown tab.
run_name = "run1" #@param {type:"string"}
include_structures = True #@param {type:"boolean"}
#@markdown <small>Untick `include_structures` to copy only the CSV and JSON summaries
#@markdown and leave the predicted structures behind — much smaller, if Drive is
#@markdown tight.</small>

import os, re, subprocess
from datetime import datetime

safe_run = re.sub(r"[^A-Za-z0-9._-]", "_", run_name) or "run1"
out_dir = f"/content/pxd_output/{safe_run}"
assert os.path.isdir(out_dir), f"no such run: {out_dir} - check run_name"
assert os.path.isfile("/content/.pxd_drive"), "Drive is not mounted - run step 2"

stamp = datetime.now().strftime("%Y%m%d_%H%M")
dest = f"{open('/content/.pxd_drive').read().strip()}/outputs/{safe_run}_{stamp}"
os.makedirs(dest, exist_ok=True)

args = ["rsync", "-a", "--info=stats1"]
if not include_structures:
    args += ["--include=*/", "--include=*.csv", "--include=*.json", "--exclude=*"]
args += [out_dir + "/", dest + "/"]
subprocess.run(args, check=True)

total = sum(os.path.getsize(os.path.join(r, f))
            for r, _, fs in os.walk(dest) for f in fs)
n = sum(len(fs) for _, _, fs in os.walk(dest))
print(f"copied {n} file(s), {total / 1e6:.1f} MB")
print(f"to: {dest.replace('/content/drive/MyDrive', 'My Drive')}")

---
## What the numbers mean

One row per designed sequence in `sample_level_output.csv`, averaged into
`summary_output.json`. The ones the filters are built on:

| Column | Reading it |
|---|---|
| `pLDDT` | AlphaFold2's confidence in the predicted complex. Higher is better. |
| `i_pTM` | Confidence in the *interface* specifically. Higher is better. |
| `i_pAE` | Expected error across the interface. **Lower** is better. |
| `af2_binder_pred_design_rmsd` | How far AF2's binder drifts from the designed backbone. **Lower** is better. |
| `bound_unbound_RMSD` | How much the binder changes shape when the target is taken away. **Lower** means it holds its fold on its own. |
| `ptx_mini_iptm_binder` | The same interface question asked of Protenix instead of AF2. Higher is better. |
| `scRMSD` / `scTM` | Monomer self-consistency: refold the designed sequence with ESMFold and compare. Lower RMSD / higher TM is better. |
| `alpha`, `beta`, `loop`, `Rg` | Secondary structure fractions and radius of gyration — shape, not quality. |
| `*_success` | 1 if that row passed every threshold in that filter, 0 if not. `af2_easy`, `af2_opt`, `ptx_mini` and the rest are defined in `pxdbench/pxd_configs/eval.py`. |

Thresholds are upstream's and are not changed here. To see or alter them, open
`/content/PXDesignBench/pxdbench/pxd_configs/eval.py` in the Files pane.

## If something fails

**Step 4 raises.** Read which check failed. NumPy pulled back to 2.x or JAX landing
on CPU both mean a dependency resolved differently than the pins intend; the fix is
*Runtime > Disconnect and delete runtime* and a fresh start, not re-running the cell
on top of a half-built environment.

**`CUDA out of memory` during step 7.** A T4 has 16 GB and this pipeline keeps
AlphaFold2 (JAX) and Protenix (PyTorch) alive on the same card. Untick
`eval_protenix_mini`, or lower `num_seqs`, or evaluate fewer structures per run with
`file_names`. A long target chain costs more than a long binder.

**It sits silent for a long time on the first run.** Expected twice: Protenix
downloads its checkpoint and CCD files (~1 GB) when it is first constructed, and
AlphaFold2 compiles its JAX graph for each new sequence length. Later designs of the
same length reuse the compilation.

**The MSA step hangs or errors.** With `orig_seqs_json` empty, Protenix asks a public
MSA server for the target's alignment, which is rate-limited and sometimes down. For
repeated work on one target, compute the MSA once and point `orig_seqs_json` at a
file naming it under `msa.precomputed_msa_dir` — the bundled
`examples/orig_seqs_test.json` is the pattern to copy.

**`Missing tool weight`.** Step 6 did not finish, or you asked for a task whose
weights you did not download — the binder pipeline needs AlphaFold2, the monomer
pipeline needs ESMFold. Re-run step 6 with the right `weights_for`.

**Nothing passes any filter.** That is a result, not a failure: the thresholds are
the ones from the PXDesign paper and real designs fail them routinely. `af2_easy` is
the lenient set; start there.

## Credit

The evaluation is [PXDesignBench](https://github.com/bytedance/PXDesignBench) by
ByteDance, installed from source and run unmodified. It stands on
[Protenix](https://github.com/bytedance/Protenix),
[ProteinMPNN](https://github.com/dauparas/ProteinMPNN),
[AlphaFold2](https://github.com/google-deepmind/alphafold) via
[ColabDesign](https://github.com/sokrypton/ColabDesign), and
[ESMFold](https://huggingface.co/facebook/esmfold_v1). Cite those, not this notebook —
the citations are listed in upstream's README.

The AlphaFold2 parameters are released under CC BY-NC 4.0: non-commercial use only.
That restriction travels with the weights, whatever the code around them is licensed as.